In [1]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [2]:
df_answers.head(3)

,question,answer_llm,answer_orig,document
0,Is it okay to join the course late if I just f...,"Yes, you can still join the course late. If yo...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still take this course even if I missed ...,"Yes, you can still join if you missed the star...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,If I join after the course has already started...,"Yes, as long as you join while the course is s...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [3]:
answers[0]

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [4]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [5]:
# judge instructions. This tells the judge what to compare and how to assign the score
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [6]:
# define the prompt template
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [7]:
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress


import os 
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
llm_model = "openai/gpt-oss-20b"

In [8]:
rec = answers[0]

In [9]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [10]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
    model =llm_model
)

eval_result

AnswerEvaluation(reasoning='The AI answer correctly conveys the same essential information as the original answer: it confirms that joining late is allowed and that a certificate requires submitting the project while submissions are still accepted. The wording is slightly different but the meaning is identical.', score='good')

In [11]:
calc_price(usage) # this is for the "gpt-5.4-mini" cost

{'input_cost': 0.00032175,
 'output_cost': 0.0007469999999999999,
 'total_cost': 0.0010687499999999998}

In [12]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [13]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"],
    model=llm_model
)

eval_result

AnswerEvaluation(reasoning='The AI answer conveys the same essential information as the original: it confirms that late enrollment is allowed and that to obtain a certificate the student must still submit a project while submissions are accepted. The phrasing and details are equivalent, so the answer is correct and complete.', score='good')

In [14]:
# Run the evaluation on all answers:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)


In [ ]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [ ]:
df_eval = pd.DataFrame(evaluations)

In [ ]:
calc_total_price(usages)

In [ ]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

In [ ]:
df_eval[df_eval["score"] == "bad"].head()

In [ ]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

In [23]:
# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
# wget -O data/rag-evaluations-new.csv ${PREFIX}/04-evaluation/data/rag-evaluations-new.csv

!cp /home/hsu/Documents/Learning/llm-zoomcamp/04-evaluation/data/rag-evaluations-new.csv data/rag-evaluations-new.csv